# Analyze evaluation output against make/model ground truth

Run `EVAL_Best_frames_against_make_model_save_outputs.py` first. This notebook loads saved artifacts instead of depending on variables left behind by an evaluation notebook.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
OUTPUT_DIR = Path("eval_outputs")
MANIFEST_PATH = OUTPUT_DIR / "latest_eval_manifest.json"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {MANIFEST_PATH}. Run EVAL_Best_frames_against_make_model_save_outputs.py first."
    )

manifest = json.loads(MANIFEST_PATH.read_text())
artifacts = manifest["artifacts"]

# Core dataframe saved by EVAL_Best_frames: one row per evaluated vehicle, enriched with GT dimensions/model/type.
target_df = pd.read_csv(artifacts["target_csv"])

# Method-specific evaluated dataframes and saved summary tables.
pred_eval_df = pd.read_csv(artifacts["pred"]["eval_csv"])
lidar_eval_df = pd.read_csv(artifacts["lidar"]["eval_csv"])
pred_summary_df = pd.read_csv(artifacts["pred"]["summary_csv"], index_col=0)
lidar_summary_df = pd.read_csv(artifacts["lidar"]["summary_csv"], index_col=0)
summary_df = pd.concat([pred_summary_df, lidar_summary_df], axis=0, sort=False)

width_used = manifest.get("width_used", "WWOM")
make_model_width_col = manifest.get("make_model_width_col", "GT_OW")
critical_dims = manifest.get("critical_dims", {})

msg = (
    f"Loaded run **{manifest['run_name']}** from `{MANIFEST_PATH}`. "
    f"Rows with GT: **{len(target_df)}**."
)
display(Markdown(msg))
display(summary_df.round(5))

In [ ]:
def numeric_copy(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def plot_width_scatter(df: pd.DataFrame, width_used: str = width_used) -> None:
    """Prediction/Lidar width vs GT width."""
    pred_col = f"PRED_{width_used}"
    cols = ["GT_OW", "LD_OW", pred_col]
    plot_df = numeric_copy(df, cols).dropna(subset=cols)

    plt.figure(figsize=(8, 8))
    plt.scatter(plot_df["GT_OW"], plot_df[pred_col], alpha=0.6, s=50, label=f"PRED_{width_used}")
    plt.scatter(plot_df["GT_OW"], plot_df["LD_OW"], alpha=0.6, s=50, label="LD_OW")

    min_val = min(plot_df["GT_OW"].min(), plot_df[pred_col].min(), plot_df["LD_OW"].min())
    max_val = max(plot_df["GT_OW"].max(), plot_df[pred_col].max(), plot_df["LD_OW"].max())
    plt.plot([min_val, max_val], [min_val, max_val], "r--", label="Perfect agreement")

    plt.title("Predicted / Lidar vs Ground Truth Vehicle Width")
    plt.xlabel("Ground Truth Width (m)")
    plt.ylabel("Predicted / Lidar Width (m)")
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.axis("equal")
    plt.show()


def metric_columns(metric: str, width_used: str = width_used) -> list[str]:
    """Return [GT, LD, PRED] columns for a metric.

    Use metric="OW" for width. The prediction width is PRED_WWOM when width_used="WWOM".
    """
    if metric == "OW":
        return ["GT_OW", "LD_OW", f"PRED_{width_used}"]
    return [f"GT_{metric}", f"LD_{metric}", f"PRED_{metric}"]


def plot_metric_histograms(df: pd.DataFrame, metric: str, bins: int = 30) -> None:
    cols = metric_columns(metric)
    existing_cols = [c for c in cols if c in df.columns]
    plot_df = numeric_copy(df, existing_cols)

    fig, axes = plt.subplots(1, len(existing_cols), figsize=(5 * len(existing_cols), 4), sharey=True)
    if len(existing_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, existing_cols):
        ax.hist(plot_df[col].dropna(), bins=bins, edgecolor="black")
        ax.set_title(f"Distribution of {col}")
        ax.set_xlabel(col)
        ax.tick_params(axis="x", rotation=60)
        ax.set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()


def metric_group(eval_df: pd.DataFrame, metric: str) -> list[str]:
    return [
        c for c in [f"DIFF_{metric}", f"ABS_DIFF_{metric}", f"ABS_DIFF_PERCENT_{metric}"]
        if c in eval_df.columns
    ]


def describe_error_metric(eval_df: pd.DataFrame, metric: str) -> pd.DataFrame:
    cols = metric_group(eval_df, metric)
    if not cols:
        raise KeyError(f"No error columns found for metric={metric!r}")
    return eval_df[cols].describe().round(3)


def save_analysis_summary(summary_df: pd.DataFrame, manifest: dict) -> Path:
    path = OUTPUT_DIR / f"{manifest['run_name']}_summary_comparison.csv"
    summary_df.to_csv(path)
    return path

## Saved summary tables

These are produced by the evaluation script and loaded here. `pred` includes wheelbase because the prediction file has `PRED_WB`; `lidar` does not include wheelbase because there is no lidar wheelbase field in the original result file.

In [ ]:
display(summary_df.round(5))
print("Prediction LaTeX:", artifacts["pred"]["summary_tex"])
print("Lidar LaTeX:", artifacts["lidar"]["summary_tex"])

## Width agreement plot

In [ ]:
plot_width_scatter(target_df, width_used=width_used)

## Dimension distributions

In [ ]:
for metric in ["OW", "OH", "OL"]:
    plot_metric_histograms(target_df, metric)

## Error distributions

Use the method-specific evaluated dataframes here. Prediction and lidar have separate `DIFF_*` columns, so the notebook no longer mutates one shared `target_df` like the original notebook did.

In [ ]:
for method_name, eval_df in [("pred", pred_eval_df), ("lidar", lidar_eval_df)]:
    display(Markdown(f"### {method_name}"))
    for metric in ["OH", "OL", width_used, "WB"]:
        if f"ABS_DIFF_{metric}" in eval_df.columns:
            display(Markdown(f"**{metric}**"))
            display(describe_error_metric(eval_df, metric))

## Breakdown by vehicle class and type

In [ ]:
# Mean absolute errors by vehicle class.
class_cols = ["class"] + [c for c in pred_eval_df.columns if c.startswith("ABS_DIFF_") and not c.startswith("ABS_DIFF_PERCENT_")]
if "class" in pred_eval_df.columns:
    class_summary = pred_eval_df[class_cols].groupby("class").agg(["count", "mean"]).round(3)
    display(class_summary)
else:
    print("No class column found.")

# Mean absolute errors by type.
type_cols = ["type"] + [c for c in pred_eval_df.columns if c.startswith("ABS_DIFF_") and not c.startswith("ABS_DIFF_PERCENT_")]
if "type" in pred_eval_df.columns:
    type_summary = pred_eval_df[type_cols].groupby("type").agg(["count", "mean"]).round(3)
    display(type_summary)
else:
    print("No type column found.")

## Save notebook-level comparison output

In [ ]:
analysis_summary_path = save_analysis_summary(summary_df, manifest)
print(f"Saved {analysis_summary_path}")